# Predictive Analytics Using Historical Data — Car Sales

**Goal:** Build a predictive model to forecast future trends using regression / time-series models. Clean and preprocess historical data, evaluate model accuracy, and visualize predictions.

This notebook builds two complementary models:
1. **Transaction-level regression** — predicts Revenue per sale from its features.
2. **Monthly revenue time-series forecast** — projects revenue for the next 3 months.

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('Car_Sales_Dataset.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

## 2. Clean & Preprocess Historical Data

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

df = df.drop_duplicates()
df = df[(df['Units Sold'] > 0) & (df['Unit Price (INR)'] > 0) & (df['Revenue (INR)'] >= 0)]
df['Month'] = df['Date'].dt.to_period('M').dt.to_timestamp()
print('\nClean shape:', df.shape)

## Part A: Transaction-Level Revenue Regression

### 3. Train/Test Split & Preprocessing Pipeline

In [ ]:
feature_cols = ['Units Sold', 'Unit Price (INR)', 'Discount (%)',
                'Brand', 'Fuel Type', 'Region', 'Dealer']
target_col = 'Revenue (INR)'

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['Units Sold', 'Unit Price (INR)', 'Discount (%)']
categorical_features = ['Brand', 'Fuel Type', 'Region', 'Dealer']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

### 4. Train Models (Linear Regression & Random Forest)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, max_depth=6)
}

results = {}
predictions = {}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    predictions[name] = preds

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

pd.DataFrame(results).T

### 5. Visualize Actual vs Predicted (Best Model)

In [ ]:
best_model_name = pd.DataFrame(results).T['R2'].idxmax()
best_preds = predictions[best_model_name]

plt.figure(figsize=(6.5, 6))
plt.scatter(y_test, best_preds, alpha=0.6, edgecolor='black')
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
plt.plot(lims, lims, '--', color='red', label='Perfect Prediction')
plt.xlabel('Actual Revenue'); plt.ylabel('Predicted Revenue')
plt.title(f'Actual vs Predicted Revenue ({best_model_name})')
plt.legend(); plt.show()

### 6. Feature Importance (Random Forest)

In [ ]:
rf_pipe = Pipeline([('prep', preprocessor), ('model', models['Random Forest'])])
rf_pipe.fit(X_train, y_train)
ohe = rf_pipe.named_steps['prep'].named_transformers_['cat']
feature_names = numeric_features + list(ohe.get_feature_names_out(categorical_features))
importances = rf_pipe.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(10)

sns.barplot(data=imp_df, x='importance', y='feature', color='#55A868')
plt.title('Top Feature Importances'); plt.show()

## Part B: Monthly Revenue Time-Series Forecast

In [ ]:
monthly = df.groupby('Month')['Revenue (INR)'].sum().reset_index().sort_values('Month').reset_index(drop=True)
monthly['t'] = np.arange(len(monthly))
monthly

In [ ]:
lr_ts = LinearRegression()
lr_ts.fit(monthly[['t']], monthly['Revenue (INR)'])
monthly['Fitted'] = lr_ts.predict(monthly[['t']])

future_t = np.arange(len(monthly), len(monthly) + 3)
future_months = pd.date_range(monthly['Month'].max() + pd.offsets.MonthBegin(1), periods=3, freq='MS')
future_preds = lr_ts.predict(future_t.reshape(-1, 1))

forecast_df = pd.DataFrame({'Month': future_months, 'Forecast_Revenue': future_preds})
forecast_df

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(monthly['Month'], monthly['Revenue (INR)'], marker='o', label='Actual Revenue')
plt.plot(monthly['Month'], monthly['Fitted'], linestyle='--', label='Trend Fit')
plt.plot(forecast_df['Month'], forecast_df['Forecast_Revenue'], marker='o', linestyle='--', label='Forecast')
plt.title('Monthly Revenue: Historical Trend & 3-Month Forecast')
plt.legend(); plt.xticks(rotation=30); plt.show()

## 7. Save Results

In [ ]:
pd.DataFrame(results).T.to_csv('model_evaluation_metrics.csv')
forecast_df.to_csv('monthly_revenue_forecast.csv', index=False)
print('Saved.')